# SPARQL Query Similarity Analysis

This notebook fetches SPARQL queries from the LSQ endpoint, analyzes duplicates, queries the DBpedia endpoint, and saves successful queries for further analysis.

### 1. Import Required Libraries

Import all necessary libraries for data fetching, processing, and similarity computations.

In [ ]:
import pandas as pd
import requests
import json
import time
import multiprocessing
from tqdm import tqdm
import ijson
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

### 2. Define SPARQL Query and Fetch Data

Set up the LSQ endpoint and query template. Define functions to run the SPARQL query with pagination and fetch results in batches.

In [ ]:
# LSQ endpoint and query template
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
QUERY_TEMPLATE = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#> 
PREFIX sd: <http://www.w3.org/ns/sparql-service-description#> 

SELECT DISTINCT ?text
WHERE {{ 
   ?query lsqv:hasRemoteExec/lsqv:endpoint <https://dbpedia.org/sparql> .
   ?query lsqv:text ?text . 
   ?query lsqv:hasSpin ?spin .
   ?spin a <http://spinrdf.org/sp#Select> . 
   ?query lsqv:hasStructuralFeatures ?features .

   FILTER NOT EXISTS {{ ?features lsqv:usesFeature lsqv:Count . }}

   FILTER(!CONTAINS(LCASE(STR(?text)), "dbpo:wikipageid"))
}}
LIMIT 1000
OFFSET {offset}
"""

def run_lsq_query(offset):
    sparql = requests.post
    from SPARQLWrapper import SPARQLWrapper, JSON
    sparql = SPARQLWrapper(LSQ_ENDPOINT)
    sparql.setQuery(QUERY_TEMPLATE.format(offset=offset))
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    return [result["text"]["value"] for result in results["results"]["bindings"]]

def fetch_all_queries(num_batches=100, batch_size=1000, delay=0.5):
    all_results = []
    for i in range(num_batches):
        offset = i * batch_size
        print(f"Fetching batch {i+1} with OFFSET {offset}...")
        try:
            rows = run_lsq_query(offset)
            all_results.extend(rows)
        except Exception as e:
            print(f"Error at offset {offset}: {e}")
        time.sleep(delay)
    return all_results

### 3. Save and Load Query Results

Save the fetched SPARQL queries to a CSV file. Load the CSV file into a pandas DataFrame for further processing.

In [ ]:
# Fetch and save queries (uncomment to run)
# queries = fetch_all_queries()
# pd.DataFrame(queries, columns=["text"]).to_csv("sparql_results.csv", index=False, encoding="utf-8")
# print("Results saved to sparql_results.csv")

# Load queries from CSV
df = pd.read_csv("sparql_results.csv")
print(f"Total queries loaded: {len(df)}")

### 4. Analyze Duplicate Queries

Analyze the DataFrame for duplicate queries, print statistics, and display sample duplicates.

In [ ]:
total_rows = len(df)
unique_rows = df["text"].nunique()
duplicate_count = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_count}")

duplicates = df[df.duplicated("text", keep=False)].sort_values("text")
print("\nSample duplicates:")
print(duplicates.head(10))

### 5. Query DBpedia Endpoint

Define a function to send SPARQL queries to the DBpedia endpoint and handle responses.

In [ ]:
def query_dbpedia(sparql_query):
    """
    Sends a SPARQL query to the DBpedia endpoint and returns the JSON result.
    Returns None if the request fails.
    """
    endpoint = "https://dbpedia.data.dice-research.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    params = {"query": sparql_query}
    try:
        response = requests.get(endpoint, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException:
        return None

### 6. Filter Successful Queries

Iterate through the loaded queries, send them to DBpedia, and collect only those with non-empty results. Save successful queries to a JSON file.

In [ ]:
successful_queries = []
failed = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Querying DBpedia"):
    sparql_query = row.get("text")
    if sparql_query:
        results = query_dbpedia(sparql_query)
        if results and "results" in results and results["results"]["bindings"]:
            successful_queries.append({
                "sparql_query": sparql_query,
                "results": results["results"]["bindings"]
            })
        else:
            failed += 1

print(f"Successful queries: {len(successful_queries)}")
print(f"Failed queries: {failed}")

with open("successful_queries.json", "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)